# Agent training in Snowball Target Environment

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/ColabNotebooks/DeepRL/UnityMLAgents

Mounted at /content/drive
/content/drive/MyDrive/ColabNotebooks/DeepRL/UnityMLAgents


In [ ]:
!pwd

/content/drive/MyDrive/ColabNotebooks/DeepRL/UnityMLAgents


Use **Unity ML-Agents Toolkit** plugin for the game engine **Unity** as an environment builder to create physical system and 3D rendering and train agents to shoot snowballs onto a spawning target, and to press a button to spawn a pyramid, navigating to it, knocking it over, and movig to the gold brick at the top.

Unity ML-Agents components:

* *Learning Environment*, contains the scene=environment and environment elements=game characters

* *Python Low-level API*, python intefrface for interacting and manipulating the environment

* *Communicator*, connects the Learning Environment(C#)  with the low level Python API

* *Python Traininer*, RL algorithms coded with PyTorch

* *Gymnasium* wrapper, encapsulates the RL envitonment

* *PetthingZoo* wrapper, the multi-agent version of the gymansium wrapper

Inside Learning Component

* *Agent*, the game charecter=agent, train by optimizing its policy callE **Brain**

* *Academy* handles  Python API requests and orchestrates agents and their decision-making

The Academil sends the order to the agents and ensures they are in sync:
* collect observations
select action using agent's policy
* take the action
*reset if reached max steps orif done

### Install Python 3.10 and pip

In [ ]:
!sudo apt-get update -y
!sudo apt-get install -y python3.10 python3.10-venv python3.10-dev python3.10-distutils
!curl -sS https://bootstrap.pypa.io/get-pip.py | python3.10
!python3.10 --version
!python3.10 -m pip --version

### Install core dependencies

In [ ]:
!python3.10 -m pip install --upgrade pip setuptools wheel
!python3.10 -m pip install torch==2.3.1 torchvision torchaudio
!python3.10 -m pip install grpcio==1.53.2 protobuf==3.20.3 pyyaml cloudpickle pettingzoo gym==0.26.2

### Clone and install ML-Agents

In [ ]:
!git clone --depth 1 https://github.com/Unity-Technologies/ml-agents

%cd ml-agents

# Install both sub-packages (envs + trainers)
!python3.10 -m pip install -e ./ml-agents-envs
!python3.10 -m pip install -e ./ml-agents

# Verify install
!python3.10 -m pip show mlagents

In [ ]:
# check the supported python version under python_requires
!cat /content/drive/MyDrive/ColabNotebooks/DeepRL/UnityMLAgents/ml-agents/ml-agents/setup.py

import os
import sys

from setuptools import setup, find_packages
from setuptools.command.install import install
from mlagents.plugins import ML_AGENTS_STATS_WRITER, ML_AGENTS_TRAINER_TYPE
import mlagents.trainers

VERSION = mlagents.trainers.__version__
EXPECTED_TAG = mlagents.trainers.__release_tag__

here = os.path.abspath(os.path.dirname(__file__))


class VerifyVersionCommand(install):
    """
    Custom command to verify that the git tag is the expected one for the release.
    Originally based on https://circleci.com/blog/continuously-deploying-python-packages-to-pypi-with-circleci/
    This differs slightly because our tags and versions are different.
    """

    description = "verify that the git tag matches our version"

    def run(self):
        tag = os.getenv("GITHUB_REF", "NO GITHUB TAG!").replace("refs/tags/", "")

        if tag != EXPECTED_TAG:
            info = "Git tag: {} does not match the expected tag of this app: {}".format(
                tag, EXPECTED_TAG
 

## <font size=16>**Snowball Target**</font>

### Prepare Unity environment

Download, unzip and move the environment linux executable zip file in ./training-envs-executables/linux/

In [ ]:
# create training-envs-executables and linux folders
!mkdir ./training-envs-executables
!mkdir ./training-envs-executables/linux

In [ ]:
# download SnowballTarget.zip using wget
!wget "https://github.com/huggingface/Snowball-Target/raw/main/SnowballTarget.zip" -O ./training-envs-executables/linux/SnowballTarget.zip


In [ ]:
# unzip the executable, -o to overide
!unzip -d ./training-envs-executables/linux/ ./training-envs-executables/linux/SnowballTarget.zip


In [ ]:
# make the file accessible
!chmod -R 755 ./training-envs-executables/linux/SnowballTarget

In [ ]:
!ls -lh ./training-envs-executables/linux/SnowballTarget/

In [ ]:
# create SnowballTarget config file and define the training hyperparameters there
#!touch config/ppo/SnowballTarget.yaml

Consider increasing max_steps: 5.0e7 and lowering learning rate

### Agent

**Julien the bear** agent hits as many targets as possible in limited time = 1000 timesteps, thus maximizing the expeced sumulative reward. It need to place itself correctly in relation to the target ans shoot to do that with a wait time = 0.5 seconds after a shot to avoid snowball spamming.

### Reward

The environment gives +1 every time the agent's snowball hits a target.

**Curiosity** through next-state prediction
Intrinsic Reward, IR = || predicted(S_t+1) - S_t+1||

Curiosity is used to push the agent to favor transitions with high prediction error, areas where the agent has spent less time, or with complex dynamics, thus ecurage to explore the environment.

### Observation Space

Observations consisit in

Multiple sets of **raycasts** are like lasers that will detect if they pass through an object:

* Back raycasts to detect invisible walls
* Front raycasts to detect
  * snowballs
  * targets
  * the rocks and invisible walls

**"Can I shoot"** boolien

### Action Space

Discrete action space consisting of

**Forward** Motion: Up/ Down
**Side** Motion: Left/Right
**Shoot snowball**

### Training

In [ ]:
# path to the Unity environment executable that the agent will interact with during training
!ls training-envs-executables/linux/SnowballTarget/SnowballTarget.x86_64

training-envs-executables/linux/SnowballTarget/SnowballTarget.x86_64


#### Capturing frames on headless Colab

In [ ]:
%%capture
!apt install python-opengl
!apt install ffmpeg
!apt install xvfb
!pip install pyvirtualdisplay
!pip install pyglet==1.5.1

In [ ]:
# install modules into the same interpreter that Colab is running, 3.12 kenel, and not the pip tied to python 3.10
!pip install pyvirtualdisplay pyglet==1.5.1

The pyvirtualdispla isinstalled under /usr/local/lib/python3.10 and is visible only to python 3.10 and not the notebook kernel which is running python 3.12

In [ ]:
# restart the kernel after installation
import os

os.kill(os.getpid(), 9)

In [ ]:
!python --version

Python 3.12.11


Force pip to install in the running kernel with sys.executable which points to the python binary of the running kernel, thus ensuring pyvirtualdisplay is installed in the same environment as the notebook uses

In [ ]:
import sys

!{sys.executable} -m pip install pyvirtualdisplay pyglet==1.5.1

  Using cached PyVirtualDisplay-3.0-py3-none-any.whl.metadata (943 bytes)
  Using cached pyglet-1.5.1-py2.py3-none-any.whl.metadata (7.6 kB)
Using cached pyglet-1.5.1-py2.py3-none-any.whl (1.0 MB)
Using cached PyVirtualDisplay-3.0-py3-none-any.whl (15 kB)


On Colab, there’s no real GPU display / monitor Unity can’t render unless a virtual display is started  --no-graphics launches in server mode, without rendering, to get frames for videoset up a firtual display xvfb before launching Unity, otherwise Unity tries to open a window and hangs

In [ ]:
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

Run environmet without  --no-graphics, so Unity will render to the virtual frambuffer and --capture-frame-rate will dump frames, png images, into the when recording frames for video set --inferance and --time-scale==1 which ensures Unity runs in real-time, smooth 30fps output without skipping or slowing down.
If changed hyperparameters like learning rate, batch size, resuming is fine.

If changed the network architecture like hidden units, layers, visual encoder, resuming may break it won’t match the checkpoint shape. In that case, start a new run-id instead of resuming.

In [ ]:
!python --version

Python 3.10.12


In [ ]:
import torch

print(torch.version.cuda)

12.6


In [ ]:
#!rm -rf results

In [ ]:
!mlagents-learn "config/ppo/SnowballTarget.yaml" \
  --env="training-envs-executables/linux/SnowballTarget/SnowballTarget.x86_64" \
  --run-id="SnowballTarget1" \
  --no-graphics

In [ ]:
# use --force if need to start the training from scratch and use the same run-id, use inference if only gathering frames for video without training
!python3.10 -m mlagents.trainers.learnconfig/ppo/SnowballTarget.yaml \
--env=training-envs-executables/linux/SnowballTarget/SnowballTarget.x86_64 \
--run-id="SnowballTarget1" \
--resume \
--inference --width=640 --height=480 --capture-frame-rate=30 --time-scale=1
# --no-graphics

#### Convert frame to video

In [ ]:
!ffmpeg -framerate 30 -pattern_type glob -i 'results/SnowballTarget1-eval/SnowballTarget/**/*.png' \
-c:v libx264 -pix_fmt yuv420p SnowballTarget-eval.mp4

In [ ]:
!ls results/SnowballTarget1/SnowballTarget

checkpoint.pt					     SnowballTarget-2112.onnx
events.out.tfevents.1759530854.bad51eb541eb.11191.0  SnowballTarget-2112.pt


In [ ]:
from IPython.display import Video

# point to video file to display it
Video("SnowballTarget-eval.mp4", embed=True, width=640, height=480)


ML-Agents saves all checkpoints in the folder results/SnowballTarget1/ppo, --resume tells ML-Agent to load the latest checkpoint in the run directory, use the same --run-id if continue training

In [ ]:
# to continue training from the last checkpoint
!mlagents-learn config/ppo/SnowballTarget.yaml \
  --env="training-envs-executables/linux/SnowballTarget/SnowballTarget" \
  --run-id="SnowballTarget1" \
  --resume \
  --no-graphics

### Push to hub

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
!ls results/SnowballTarget1

configuration.yaml  run_logs  SnowballTarget  SnowballTarget.onnx


In [ ]:
!mlagents-push-to-hf \
--run-id="SnowballTarget1" \
--local-dir="./results/SnowballTarget1" \
--repo-id="elliemci/ppo-Snowball-Target" \
--commit-message="add video"

[INFO] This function will create a model card and upload your SnowballTarget1 into HuggingFace Hub. This is a work in progress: If you encounter a bug, please send open an issue
[INFO] Pushing repo SnowballTarget1 to the Hugging Face Hub
Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  .../SnowballTarget-2112.onnx:  15% 94.5k/650k [00:00<?, ?B/s]


  ...et/SnowballTarget-2112.pt:  15% 187k/1.29M [00:00<?, ?B/s]



  ...wballTarget/checkpoint.pt:  15% 187k/1.29M [00:00<?, ?B/s]




  ...8804.bad51eb541eb.45140.0:  15% 162/1.11k [00:00<?, ?B/s]





  ...rget1/SnowballTarget.onnx:  15% 94.5k/650k [00:00<?, ?B/s]

  .../SnowballTarget-2112.onnx:  15% 94.5k/650k [00:00<?, ?B/s]


  ...et/SnowballTarget-2112.pt:  15% 187k/1.29M [00:00<?, ?B/s]



  ...wballTarget/checkpoint.pt:  15% 187k/1.29M [00:00<?, ?B/s]




  ...8804.bad51eb541eb.45140.0:  15% 162/1.11k [00:00<?, ?B/s]





Processing F

To visualize the agent online:

1. Go to https://huggingface.co/spaces/ThomasSimonini/ML-Agents-SnowballTarget

  * type username: elliemci
  * choose model repository: ppo-Snowball-Target
  * select model *.nn/*.onnx file

2. Launch the game and put it in full screen by clicking on the bottom right button

## <font size=18>**Pyramids**</font>

In [ ]:
# download Pyramids.zip
!wget "https://huggingface.co/spaces/unity/ML-Agents-Pyramids/resolve/main/Pyramids.zip" -O ./training-envs-executables/linux/Pyramids.zip

In [ ]:
# unzip
!unzip -d ./training-envs-executables/linux/ ./training-envs-executables/linux/Pyramids.zip

In [ ]:
# make it accessible
!chmod -R 755 ./training-envs-executables/linux/Pyramids/Pyramids

Pyramids environmet is made by Unity, config file already existis in ./content/ml-agents/config/ppo/PyramidsRND.yaml, where RND stands for random network distilation, a way to generate curiosity rewards. Modify total trainig steps to hit benchmark of mean reward = 1.75 in only 1M training steps

### Training

In [ ]:
!mlagents-learn ./config/ppo/PyramidsRND.yaml \
--env=./training-envs-executables/linux/Pyramids/Pyramids \
--run-id="PyramidsTraining" \
--no-graphics


            ┐  ╖
        ╓╖╬│╡  ││╬╖╖
    ╓╖╬│││││┘  ╬│││││╬╖
 ╖╬│││││╬╜        ╙╬│││││╖╖                               ╗╗╗
 ╬╬╬╬╖││╦╖        ╖╬││╗╣╣╣╬      ╟╣╣╬    ╟╣╣╣             ╜╜╜  ╟╣╣
 ╬╬╬╬╬╬╬╬╖│╬╖╖╓╬╪│╓╣╣╣╣╣╣╣╬      ╟╣╣╬    ╟╣╣╣ ╒╣╣╖╗╣╣╣╗   ╣╣╣ ╣╣╣╣╣╣ ╟╣╣╖   ╣╣╣
 ╬╬╬╬┐  ╙╬╬╬╬│╓╣╣╣╝╜  ╫╣╣╣╬      ╟╣╣╬    ╟╣╣╣ ╟╣╣╣╙ ╙╣╣╣  ╣╣╣ ╙╟╣╣╜╙  ╫╣╣  ╟╣╣
 ╬╬╬╬┐     ╙╬╬╣╣      ╫╣╣╣╬      ╟╣╣╬    ╟╣╣╣ ╟╣╣╬   ╣╣╣  ╣╣╣  ╟╣╣     ╣╣╣┌╣╣╜
 ╬╬╬╜       ╬╬╣╣      ╙╝╣╣╬      ╙╣╣╣╗╖╓╗╣╣╣╜ ╟╣╣╬   ╣╣╣  ╣╣╣  ╟╣╣╦╓    ╣╣╣╣╣
 ╙   ╓╦╖    ╬╬╣╣   ╓╗╗╖            ╙╝╣╣╣╣╝╜   ╘╝╝╜   ╝╝╝  ╝╝╝   ╙╣╣╣    ╟╣╣╣
   ╩╬╬╬╬╬╬╦╦╬╬╣╣╗╣╣╣╣╣╣╣╝                                             ╫╣╣╣╣
      ╙╬╬╬╬╬╬╬╣╣╣╣╣╣╝╜
          ╙╬╬╬╣╣╣╜
             ╙
        
 Version information:
  ml-agents: 1.2.0.dev0,
  ml-agents-envs: 1.2.0.dev0,
  Communicator API: 1.5.0,
  PyTorch: 2.8.0+cu128
[INFO] Connected to Unity environment with package version 2.2.1-exp.1 and communication version 1.5.0
[INFO] Connected new brain: Pyramids?team=0

In [ ]:
!pwd

/content/drive/MyDrive/ColabNotebooks/DeepRL/UnityMLAgents/ml-agents


### Push to hub

In [ ]:
!ls results

'Pyramids Training'   SnowballTarget1


In [ ]:
!mlagents-push-to-hf \
--run-id="Pyramids Training" \
--local-dir="./results/PyramidsTraining" \
--repo-id="elliemci/ppo-Pyramids" \
--commit-message="First training of Pyramids"

[INFO] This function will create a model card and upload your Pyramids Training into HuggingFace Hub. This is a work in progress: If you encounter a bug, please send open an issue
[INFO] Pushing repo Pyramids Training to the Hugging Face Hub
Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  ...ids/Pyramids-1499952.onnx:   1% 12.9k/1.42M [00:00<?, ?B/s]


  ...amids/Pyramids-1499952.pt:   1% 78.6k/8.66M [00:00<?, ?B/s]



  ...ids/Pyramids-1999968.onnx:   1% 12.9k/1.42M [00:00<?, ?B/s]




  ...amids/Pyramids-1999968.pt:   1% 78.6k/8.66M [00:00<?, ?B/s]





  ...ids/Pyramids-2499979.onnx:   1% 12.9k/1.42M [00:00<?, ?B/s]






  ...amids/Pyramids-2499979.pt:   1% 78.6k/8.66M [00:00<?, ?B/s]







  ...ids/Pyramids-2999878.onnx:   1% 12.9k/1.42M [00:00<?, ?B/s]

... [921 lines truncated] ...






  ...ng/Pyramids/checkpoint.pt: 100% 8.66M/8.66M [00:02<00:00, 3.57MB/s]









  ...9209.